<a href="https://colab.research.google.com/github/divy-brahmbhatt/distilbert-live-sentiment-etl/blob/main/ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pandas requests plotly

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class SentimentEngine:
  def __init__(self):
    self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Loading DistilBERT Engine on: {self.device}")

    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)

    self.model.eval()

  def predict(self, text:str) -> dict:
    if not text or not text.strip():
      return {"label": "NEUTRAL", "score":0.0}

    inputs = self.tokenizer(text, return_tensors="pt", truncation = True, max_length = 512).to(self.device)

    with torch.no_grad():
      outputs = self.model(**inputs)
      probs = torch.softmax(outputs.logits, dim=-1)

    label_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][label_id].item()

    labels = ["NEGATIVE", "POSITIVE"]
    return {
        "label": labels[label_id],
        "score": round(confidence, 4)
    }

In [20]:
import requests
import pandas as pd
import datetime

def fetch_live_headlines(limit:int = 15) -> pd.DataFrame:
  NEWS_FEED_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
  ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

  try:
    response = requests.get(NEWS_FEED_URL, timeout=5)
    story_ids = response.json()[:limit]

    stories = []
    for sid in story_ids:
      item_res = requests.get(ITEM_URL.format(sid), timeout = 3)
      item = item_res.json()

      if item and "title" in item:
        stories.append({
                    "id": item.get("id"),
                    "title": item.get("title"),
                    "score": item.get("score", 0),
                    "timestamp": datetime.datetime.fromtimestamp(item.get("time", 0)).strftime('%Y-%m-%d %H:%M:%S')
                })

    return pd.DataFrame(stories)
  except Exception as e:
    print(f"Error fetching headlines: {e}")
    return pd.DataFrame([])

In [22]:
import plotly.express as px

engine = SentimentEngine()

df = fetch_live_headlines(limit=15)

preds = [engine.predict(title) for title in df["title"].tolist()]

df["sentiment_label"] = [p["label"] for p in preds]
df["confident_score"] = [p["score"] for p in preds]

Loading DistilBERT Engine on: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [23]:
fig = px.histogram(
    df,
    x="sentiment_label",
    color="sentiment_label",
    title="Real-Time Headline Sentiment Distribution (DistilBERT Analysis)",
    color_discrete_map={"POSITIVE": "#2ecc71", "NEGATIVE": "#e74c3c"}
)

fig.show()